# Ensemble surrogate + constrained Bayesian optimization

This notebook builds a pure NumPy/SciPy bootstrap random-Fourier-feature ensemble for the four structural outputs, then uses the ensemble uncertainty in a constrained expected-improvement search. The supplied L/D surrogate is evaluated directly.

The structural evaluation is **surrogate-only**: no callable nTop/FE solver is included in this repository. Final designs are therefore repaired and revalidated against every ensemble member, not claimed as new FE results.

In [1]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from scipy.special import ndtr
from scipy.stats import qmc

ROOT = Path.cwd()
if not (ROOT / 'data' / 'bwb_structures_dataset.csv').exists():
    raise FileNotFoundError('Run this notebook from the repository root.')
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT / 'models' / 'ld_surrogate'))
from predict_ld import predict_ld_batch

RNG_SEED = 20260819
N_ENSEMBLE = 8
N_RFF = 192
STRESS_LIMIT_MPA = 335.0
MASS_REFERENCE_KG = 50.0

PLANFORM = ['C2/C1', 'C3/C1', 'C4/C1', 'B1/C1', 'B2/C1', 'B3/C1', 'X3/C1', 'S1', 'S3', 'C1']
STRUCTURE = ['Skin Thickness', 'Front Spar Chord %', 'Rear Spar Chord %', 'Spar Thickness', '# of Ribs',
             'Rib Thickness', 'Wingbox Cutout', '# of Fuselage Ribs', '# of Fuselage Spars',
             'Fuselage Struct Thickness', 'Fuselage Struct Width']
DESIGN_COLUMNS = PLANFORM + STRUCTURE
FLIGHT_COLUMNS = ['Altitude', 'KCAS', 'AOA']
INPUT_COLUMNS = DESIGN_COLUMNS + FLIGHT_COLUMNS
TARGET_COLUMNS = ['Aircraft Empty Weight', 'Payload Volume', 'Fuel Volume', 'Max Hotspot Stress']
TARGET_LABELS = ['empty_mass_kg', 'payload_volume_m3', 'fuel_volume_m3', 'max_hotspot_stress_mpa']

TEST_CASES = [
    dict(name='High Speed Dash', ld_target=6.0, payload_volume_min_m3=0.75, fuel_volume_min_m3=0.45, altitude_kft=15.0, kcas_kt=120.0, aoa_deg=1.0),
    dict(name='Max Endurance', ld_target=10.0, payload_volume_min_m3=0.80, fuel_volume_min_m3=0.45, altitude_kft=15.0, kcas_kt=45.0, aoa_deg=8.0),
    dict(name='Max Capacity', ld_target=15.0, payload_volume_min_m3=1.00, fuel_volume_min_m3=0.65, altitude_kft=5.0, kcas_kt=220.0, aoa_deg=4.5),
]

BOUNDS = np.array([
    [0.55, 0.85], [0.18, 0.28], [0.06, 0.09], [0.10, 0.20], [0.05, 0.20], [0.35, 0.70], [0.50, 0.65], [40, 60], [20, 40], [2500, 4000],
    [0.0003, 0.0050], [0.18, 0.35], [0.55, 0.75], [0.00098, 0.0080], [3, 14], [0.0015, 0.015], [0.01, 0.05], [0, 4], [3, 12], [0.002, 0.025], [0.0010, 0.015],
], dtype=float)
PHYSICAL_BOUNDS = BOUNDS.copy()
PHYSICAL_BOUNDS[17] = [3, 11]  # repaired fuselage-rib values, not the latent 0..4 optimizer index

df = pd.read_csv(ROOT / 'data' / 'bwb_structures_dataset.csv')
# Remove divergent elastic-solve artifacts documented in data/README.md.
df = df.loc[df['Max Hotspot Stress'] < 1e4].reset_index(drop=True)
print(f'Using {len(df):,} structural rows after artifact filtering.')

Using 13,597 structural rows after artifact filtering.


In [2]:
def repair_design(x):
    """Clip a 21-D optimizer vector and enforce discrete design axes."""
    z = np.clip(np.asarray(x, dtype=float), BOUNDS[:, 0], BOUNDS[:, 1]).copy()
    z[14] = np.rint(z[14])                         # # of Ribs
    z[17] = 3 + 2 * np.rint(z[17])                # latent 0..4 -> odd fuselage ribs
    z[18] = np.rint(z[18])                         # # of Fuselage Spars
    return z

def to_model_input(designs, mission):
    designs = np.atleast_2d(np.asarray(designs, dtype=float))
    flight = np.tile([mission['altitude_kft'], mission['kcas_kt'], mission['aoa_deg']], (len(designs), 1))
    return np.hstack([designs, flight])

def output_transform(frame):
    y = np.column_stack([
        frame['Aircraft Empty Weight'].to_numpy(float),
        frame['Payload Volume'].to_numpy(float) / 1e9,
        frame['Fuel Volume'].to_numpy(float) / 1e9,
        frame['Max Hotspot Stress'].to_numpy(float),
    ])
    return np.log(np.maximum(y, 1e-12))

rng = np.random.default_rng(RNG_SEED)
feasible = (df['Max Hotspot Stress'].to_numpy() <= STRESS_LIMIT_MPA)
train_indices, test_indices = [], []
for group in (np.flatnonzero(feasible), np.flatnonzero(~feasible)):
    shuffled = rng.permutation(group)
    cut = int(0.8 * len(shuffled))
    train_indices.extend(shuffled[:cut]); test_indices.extend(shuffled[cut:])
train_indices, test_indices = np.array(train_indices), np.array(test_indices)

X_all = df[INPUT_COLUMNS].to_numpy(float)
X_mean, X_scale = X_all[train_indices].mean(axis=0), X_all[train_indices].std(axis=0)
X_scale[X_scale == 0] = 1.0
X_train = (X_all[train_indices] - X_mean) / X_scale
X_test = (X_all[test_indices] - X_mean) / X_scale
Y_all_log = output_transform(df)
Y_train, Y_test = Y_all_log[train_indices], Y_all_log[test_indices]
print(f'Train rows: {len(train_indices):,}; holdout rows: {len(test_indices):,}.')

Train rows: 10,877; holdout rows: 2,720.


In [3]:
class RFFRidgeEnsemble:
    """Bootstrap RBF-kernel approximation with multi-output ridge regression."""
    def __init__(self, n_members=N_ENSEMBLE, n_features=N_RFF, seed=RNG_SEED):
        self.n_members, self.n_features, self.seed = n_members, n_features, seed
        self.members = []

    def fit(self, X, Y):
        rng = np.random.default_rng(self.seed)
        self.members = []
        for member in range(self.n_members):
            indices = rng.integers(0, len(X), size=len(X))
            length_scale = rng.uniform(1.4, 3.0)
            ridge = 10 ** rng.uniform(-2.5, -1.2)
            W = rng.normal(scale=1.0 / length_scale, size=(X.shape[1], self.n_features))
            b = rng.uniform(0, 2 * np.pi, size=self.n_features)
            phi = np.sqrt(2.0 / self.n_features) * np.cos(X[indices] @ W + b)
            y_mean = Y[indices].mean(axis=0)
            beta = np.linalg.solve(phi.T @ phi + ridge * np.eye(self.n_features), phi.T @ (Y[indices] - y_mean))
            self.members.append((W, b, beta, y_mean))
        return self

    def predict_members_log(self, X):
        X = np.atleast_2d(X)
        values = []
        for W, b, beta, y_mean in self.members:
            phi = np.sqrt(2.0 / self.n_features) * np.cos(X @ W + b)
            values.append(phi @ beta + y_mean)
        return np.stack(values, axis=0)

    def predict_members(self, X):
        return np.exp(self.predict_members_log(X))

    def predict(self, X):
        values = self.predict_members(X)
        return values.mean(axis=0), values.std(axis=0, ddof=1)

ensemble = RFFRidgeEnsemble().fit(X_train, Y_train)
test_mean, test_std = ensemble.predict(X_test)
test_actual = np.exp(Y_test)
validation = pd.DataFrame({
    'output': TARGET_LABELS,
    'mae': np.mean(np.abs(test_mean - test_actual), axis=0),
    'rmse': np.sqrt(np.mean((test_mean - test_actual) ** 2, axis=0)),
})
stress_truth = test_actual[:, 3] <= STRESS_LIMIT_MPA
stress_pred = test_mean[:, 3] + 2 * test_std[:, 3] <= STRESS_LIMIT_MPA
print(validation.to_string(index=False, float_format=lambda x: f'{x:.4g}'))
print(f'Conservative stress-feasibility accuracy: {(stress_truth == stress_pred).mean():.3f}')

                output     mae    rmse
         empty_mass_kg      48   116.1
     payload_volume_m3  0.1259  0.1746
        fuel_volume_m3 0.05142 0.07502
max_hotspot_stress_mpa   331.9   835.2
Conservative stress-feasibility accuracy: 0.665


In [4]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, label, actual, predicted in zip(axes.flat, TARGET_LABELS, test_actual.T, test_mean.T):
    lo, hi = np.quantile(np.r_[actual, predicted], [0.01, 0.99])
    ax.scatter(actual, predicted, s=7, alpha=0.28, color='tab:blue')
    ax.plot([lo, hi], [lo, hi], '--', color='black', lw=1)
    ax.set(xlabel='holdout FE label', ylabel='ensemble prediction', title=label)
fig.suptitle('Structural ensemble holdout validation')
fig.tight_layout()
fig.savefig(OUTPUTS / 'model_validation.png', dpi=180, bbox_inches='tight')
plt.show()

def ld_for_designs(designs, mission):
    frame = pd.DataFrame(np.atleast_2d(designs), columns=DESIGN_COLUMNS)
    return predict_ld_batch(frame, mission['altitude_kft'], mission['kcas_kt'], mission['aoa_deg'])

def base_loss(mass, ld, payload, fuel, mission):
    return (0.4 * mass / MASS_REFERENCE_KG
            + 0.2 * np.maximum(0.0, (mission['ld_target'] - ld) / mission['ld_target'])
            + 0.2 * np.maximum(0.0, (mission['fuel_volume_min_m3'] - fuel) / mission['fuel_volume_min_m3'])
            + 0.2 * np.maximum(0.0, (mission['payload_volume_min_m3'] - payload) / mission['payload_volume_min_m3']))

def predict_designs(designs, mission):
    repaired = np.vstack([repair_design(x) for x in np.atleast_2d(designs)])
    X = (to_model_input(repaired, mission) - X_mean) / X_scale
    member_structural = ensemble.predict_members(X)  # member, candidate, output
    ld = ld_for_designs(repaired, mission)
    member_losses = base_loss(member_structural[:, :, 0], ld[None, :], member_structural[:, :, 1], member_structural[:, :, 2], mission)
    return repaired, ld, member_structural, member_losses

def optimize_mission(mission, seed):
    # A Sobol set provides a deterministic reference incumbent for expected improvement.
    initial = qmc.Sobol(d=len(BOUNDS), scramble=True, seed=seed).random_base2(m=9)
    initial = BOUNDS[:, 0] + initial * (BOUNDS[:, 1] - BOUNDS[:, 0])
    repaired, ld, structural, losses = predict_designs(initial, mission)
    stress_mean, stress_std = structural[:, :, 3].mean(axis=0), structural[:, :, 3].std(axis=0, ddof=1)
    conservative_feasible = stress_mean + 2 * stress_std <= STRESS_LIMIT_MPA
    loss_mean = losses.mean(axis=0)
    incumbent = loss_mean[conservative_feasible].min() if conservative_feasible.any() else loss_mean.min()

    def negative_acquisition(x):
        repaired, ld, structural, losses = predict_designs(np.asarray(x)[None, :], mission)
        mu, sigma = losses[:, 0].mean(), max(losses[:, 0].std(ddof=1), 1e-8)
        z = (incumbent - mu) / sigma
        expected_improvement = (incumbent - mu) * ndtr(z) + sigma * np.exp(-0.5 * z * z) / np.sqrt(2 * np.pi)
        stress_mu, stress_sigma = structural[:, 0, 3].mean(), max(structural[:, 0, 3].std(ddof=1), 1e-8)
        feasible_probability = ndtr((STRESS_LIMIT_MPA - stress_mu) / stress_sigma)
        return -float(expected_improvement * feasible_probability + 1e-12 * feasible_probability)

    result = differential_evolution(negative_acquisition, bounds=BOUNDS.tolist(), seed=seed, maxiter=35, popsize=7, tol=1e-5, polish=False, updating='immediate')
    # Final selection is made *after repair*: prefer the lowest-loss conservative-feasible point
    # among both the Sobol reference set and the constrained-acquisition proposal.
    final_pool = np.vstack([initial, result.x])
    repaired, ld, structural, losses = predict_designs(final_pool, mission)
    mean_stress = structural[:, :, 3].mean(axis=0)
    std_stress = structural[:, :, 3].std(axis=0, ddof=1)
    final_losses = losses.mean(axis=0)
    feasible_pool = mean_stress + 2 * std_stress <= STRESS_LIMIT_MPA
    selected = (np.flatnonzero(feasible_pool)[np.argmin(final_losses[feasible_pool])]
                if feasible_pool.any() else np.argmin(mean_stress + 2 * std_stress))
    return repaired[selected], float(ld[selected]), structural[:, selected, :].mean(axis=0), structural[:, selected, :].std(axis=0, ddof=1), float(final_losses[selected]), -float(result.fun), float(incumbent)

/var/folders/g6/r3kn9bqj7yq3hvs90h5jrxn00000gn/T/ipykernel_79825/4095018576.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
def validate_final_design(design, mission):
    repaired, ld, members, member_losses = predict_designs(np.asarray(design)[None, :], mission)
    x = repaired[0]
    in_bounds = bool(np.all(x >= PHYSICAL_BOUNDS[:, 0]) and np.all(x <= PHYSICAL_BOUNDS[:, 1]))
    discrete_valid = bool(float(x[14]).is_integer() and float(x[18]).is_integer() and int(x[17]) in {3, 5, 7, 9, 11})
    mean, std = members[:, 0, :].mean(axis=0), members[:, 0, :].std(axis=0, ddof=1)
    stress_ucb = mean[3] + 2 * std[3]
    return x, float(ld[0]), mean, std, float(stress_ucb), in_bounds, discrete_valid, bool(stress_ucb <= STRESS_LIMIT_MPA), float(member_losses[:, 0].mean())

records = []
for case_number, mission in enumerate(TEST_CASES, start=1):
    raw_design, _, _, _, _, acquisition, incumbent = optimize_mission(mission, RNG_SEED + case_number)
    design, ld, mean, std, stress_ucb, in_bounds, discrete_valid, stress_feasible, loss = validate_final_design(raw_design, mission)
    record = {
        'case': case_number, 'mission': mission['name'], **mission, **dict(zip(DESIGN_COLUMNS, design)),
        'ld': ld, 'empty_mass_kg_pred': mean[0], 'empty_mass_kg_std': std[0],
        'payload_volume_m3_pred': mean[1], 'payload_volume_m3_std': std[1],
        'fuel_volume_m3_pred': mean[2], 'fuel_volume_m3_std': std[2],
        'max_hotspot_stress_mpa_pred': mean[3], 'max_hotspot_stress_mpa_std': std[3],
        'stress_upper_confidence_mpa': stress_ucb, 'in_bounds': in_bounds, 'discrete_valid': discrete_valid,
        'conservative_stress_feasible': stress_feasible, 'surrogate_predicted_loss': loss,
        'acquisition_value': acquisition, 'initial_incumbent_loss': incumbent,
    }
    records.append(record)

results = pd.DataFrame(records)
csv_path = OUTPUTS / 'constrained_bo_results.csv'
results.to_csv(csv_path, index=False)
display_columns = ['mission', 'surrogate_predicted_loss', 'ld', 'empty_mass_kg_pred', 'payload_volume_m3_pred', 'fuel_volume_m3_pred', 'max_hotspot_stress_mpa_pred', 'stress_upper_confidence_mpa', 'conservative_stress_feasible']
print(results[display_columns].round(4).to_string(index=False))
print(f'Wrote {csv_path.relative_to(ROOT)}')

        mission  surrogate_predicted_loss      ld  empty_mass_kg_pred  payload_volume_m3_pred  fuel_volume_m3_pred  max_hotspot_stress_mpa_pred  stress_upper_confidence_mpa  conservative_stress_feasible
High Speed Dash                    0.9725  7.3666             98.5111                  0.5175               0.1746                     154.1232                     343.8547                         False
  Max Endurance                    1.1084  7.6322            107.3971                  0.4683               0.1823                     150.4600                     231.5371                          True
   Max Capacity                    1.2622 12.2319            122.7881                  0.5114               0.1780                     222.0703                     848.0041                         False
Wrote outputs/constrained_bo_results.csv


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
names = results['mission'].tolist()
x = np.arange(len(results)); width = 0.25
axes[0].bar(x - width, results['ld'], width, label='L/D predicted')
axes[0].bar(x, results['payload_volume_m3_pred'], width, label='payload m³ predicted')
axes[0].bar(x + width, results['fuel_volume_m3_pred'], width, label='fuel m³ predicted')
axes[0].set(xticks=x, xticklabels=names, title='Final performance predictions')
axes[0].tick_params(axis='x', rotation=18)
axes[0].legend(fontsize=8)

axes[1].bar(names, results['max_hotspot_stress_mpa_pred'], yerr=2 * results['max_hotspot_stress_mpa_std'], capsize=5, label='mean ± 2σ')
axes[1].axhline(STRESS_LIMIT_MPA, color='tab:red', linestyle='--', label='335 MPa limit')
axes[1].scatter(names, results['stress_upper_confidence_mpa'], color='black', zorder=3, label='conservative bound')
axes[1].set(ylabel='hotspot stress (MPa)', title='Post-repair surrogate stress validation')
axes[1].tick_params(axis='x', rotation=18)
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUTS / 'optimization_summary.png', dpi=180, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(names, results['acquisition_value'], label='selected constrained EI')
ax.scatter(names, results['initial_incumbent_loss'], color='tab:red', zorder=3, label='Sobol incumbent loss')
ax.set(ylabel='acquisition / loss scale', title='Constrained acquisition diagnostics')
ax.tick_params(axis='x', rotation=18)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUTS / 'acquisition_diagnostics.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved model_validation.png, optimization_summary.png, and acquisition_diagnostics.png in outputs/.')

Saved model_validation.png, optimization_summary.png, and acquisition_diagnostics.png in outputs/.


/var/folders/g6/r3kn9bqj7yq3hvs90h5jrxn00000gn/T/ipykernel_79825/2679876890.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/g6/r3kn9bqj7yq3hvs90h5jrxn00000gn/T/ipykernel_79825/2679876890.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
